# Custom (YouTube-scraped) VITS Model Evaluation
`voicelk_vits_custom-September-08-2026_05+59AM-fd04a0a`

Mee notebook ekin evaluate karanne **custom** dataset ekin (YouTube-scraped, single real speaker,
IPA text pre-computed) train karapu VITS checkpoint ekayi. Pathnirwana runs (A/B/C) walata wada mek godak wenas:

- **Single speaker, no speaker-embedding table** (`use_speaker_embedding=False`, `num_speakers=0`) --
  pathnirwana runs wala tibuna 234-bogus-speaker bug ekata mulika washayenma exposure ekak nae.
- **`custom_formatter`** dhanatamath `line.split("|", 1)` paavichchi karanawa (outside-in split) --
  embedded `|` character issue ekin arakshithayi.
- **16 kHz sample rate**, `mel_fmax=8000` (pathnirwana wala 22.05 kHz, `mel_fmax=None`) -- audio bandwidth wenas.
- **`num_chars=90`** (pathnirwanata wada dhegunayakata wada wadi) -- heavy Sinhala/English code-switching
  nisa IPA character set eka wadaa pulul.
- **Ground-truth `.wav` files locally available** (`TTS/data/wavs`, files 908k, metadata line 860ma matches) --
  pathnirwana runs waladi mek nothibuna nisa MCD ganayanaya karanna baeri wuna; mee notebook ekedi
  **atthatama MCD ganayanaya karanna puluwan**.
- Trained on **Kaggle** (not Colab), single GPU, 7 tfevents files (session reconnects).

**Kernel:** mee notebook eka `venv_evoluation` (`D:\RUSL\Final Project\TTS\voicelk_ml\venv_evoluation`)
environment ekin run karanna. Eke torch 2.14, tensorboard 2.21, librosa 1.0, soundfile, pandas, matplotlib,
scipy, saha vendored-TTS runtime deps (coqpit, pysbd, anyascii, sinling, eng_to_ipa, mutagen, inflect)
dhanatamath install karala thiyenawa.

Cells **top to bottom anupilivelinma** run karanna -- pasubima cells kalin cells wala variables matha
rendha pawathinawa (`model`, `text_pipeline`, `scalars_df` wage).


---
## 1. Setup and paths

In [15]:
import os, sys, json, time, glob, hashlib
import pandas as pd
import numpy as np

VOICELK_ML = r"D:\RUSL\Final Project\TTS\voicelk_ml"
TTS_ROOT = r"D:\RUSL\Final Project\TTS"
MODELS_DIR = os.path.join(TTS_ROOT, "Models")
OUT_DIR = os.path.join(VOICELK_ML, "model_evaluate", "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# Vendored TTS source (NOT pip-installed) -- see model_training/requirements.txt
sys.path.insert(0, os.path.join(VOICELK_ML, "model_training"))
sys.path.insert(0, os.path.join(VOICELK_ML, "model_engine"))

RUN = os.path.join(MODELS_DIR, "voicelk_vits_custom-September-08-2026_05+59AM-fd04a0a")
RUN_CONFIG = os.path.join(RUN, "config.json")

# Ground-truth audio for this dataset lives at the PROJECT ROOT data/ folder
# (not voicelk_ml/data/, which only holds metadata copies locally).
GT_DATA_DIR = os.path.join(TTS_ROOT, "data")
GT_WAV_DIR = os.path.join(GT_DATA_DIR, "wavs")
CUSTOM_METADATA = os.path.join(VOICELK_ML, "data", "custom_metadata.txt")

with open(RUN_CONFIG, encoding="utf-8") as f:
    cfg = json.load(f)

ma = cfg["model_args"]
print("run_name               :", cfg.get("run_name"))
print("num_speakers            :", ma.get("num_speakers"))
print("use_speaker_embedding   :", ma.get("use_speaker_embedding"))
print("num_chars                :", ma.get("num_chars"))
print("sample_rate              :", cfg["audio"].get("sample_rate"))
print("mel_fmax                 :", cfg["audio"].get("mel_fmax"))
print("epochs / batch_size      :", cfg.get("epochs"), "/", cfg.get("batch_size"))
print("lr / lr_gen / lr_disc    :", cfg.get("lr"), cfg.get("lr_gen"), cfg.get("lr_disc"))
print("mixed_precision          :", cfg.get("mixed_precision"))
print("GT wav dir exists        :", os.path.isdir(GT_WAV_DIR), "-", GT_WAV_DIR)
print("custom_metadata exists   :", os.path.isfile(CUSTOM_METADATA))


run_name               : voicelk_vits_custom
num_speakers            : 0
use_speaker_embedding   : False
num_chars                : 90
sample_rate              : 16000
mel_fmax                 : 8000
epochs / batch_size      : 1000 / 16
lr / lr_gen / lr_disc    : 0.001 0.0002 0.0002
mixed_precision          : True
GT wav dir exists        : True - D:\RUSL\Final Project\TTS\data\wavs
custom_metadata exists   : True


---
## 2. Checkpoint inventory & model size

Hama checkpoint ekakma (`best_model.pth`, `best_model_51837.pth`, periodic `checkpoint_*.pth` 4ma) `torch.load` karala, eke stored `step`/`epoch`/`date`/`model_loss` kiyawala, parameter count ekath ganayanaya karanawa. `best_model.pth` saha `best_model_51837.pth` byte-identical da kiyala MD5 ekin check karanawa (Run C ekedi mee deka identical wuna -- methanath eheme wenawada balamu).

In [16]:
def md5_of_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

import torch

CKPT_NAMES = ["best_model.pth", "best_model_51837.pth", "checkpoint_20000.pth",
              "checkpoint_30000.pth", "checkpoint_40000.pth", "checkpoint_50000.pth"]

ckpt_rows = []
for name in CKPT_NAMES:
    path = os.path.join(RUN, name)
    if not os.path.isfile(path):
        print("MISSING:", name)
        continue
    state = torch.load(path, map_location="cpu", weights_only=False)
    n_params = sum(v.numel() for v in state["model"].values() if hasattr(v, "numel"))
    row = {
        "checkpoint": name,
        "size_mb": round(os.path.getsize(path) / 1024**2, 1),
        "step": state.get("step"),
        "epoch": state.get("epoch"),
        "date": state.get("date"),
        "n_params": n_params,
    }
    if "model_loss" in state and state["model_loss"] is not None:
        row["train_loss"] = state["model_loss"].get("train_loss")
        row["eval_loss"] = state["model_loss"].get("eval_loss")
    ckpt_rows.append(row)
    extra = f" model_loss={state.get('model_loss')}" if state.get("model_loss") else ""
    print(f"{name:26s} size={row['size_mb']}MB step={row['step']} epoch={row['epoch']} "
          f"date={row['date']} n_params={n_params:,}{extra}")

ckpt_df = pd.DataFrame(ckpt_rows)
ckpt_df.to_csv(os.path.join(OUT_DIR, "run_custom_checkpoint_metadata.csv"), index=False)

md5_best = md5_of_file(os.path.join(RUN, "best_model.pth"))
md5_51837 = md5_of_file(os.path.join(RUN, "best_model_51837.pth"))
print("\nMD5 best_model.pth        :", md5_best)
print("MD5 best_model_51837.pth  :", md5_51837)
print("Identical?", md5_best == md5_51837)

BEST_CKPT = os.path.join(RUN, "best_model.pth")


best_model.pth             size=951.6MB step=51837 epoch=33 date=September 09, 2026 n_params=83,051,308 model_loss={'train_loss': 17.980284724595414, 'eval_loss': 18.221116065979004}
best_model_51837.pth       size=951.6MB step=51837 epoch=33 date=September 09, 2026 n_params=83,051,308 model_loss={'train_loss': 17.980284724595414, 'eval_loss': 18.221116065979004}
checkpoint_20000.pth       size=951.6MB step=20000 epoch=291 date=September 08, 2026 n_params=83,051,308 model_loss={'train_loss': 18.079256653785706, 'eval_loss': None}
checkpoint_30000.pth       size=951.6MB step=30000 epoch=185 date=September 08, 2026 n_params=83,051,308 model_loss={'train_loss': 17.885717749595642, 'eval_loss': None}
checkpoint_40000.pth       size=951.6MB step=40000 epoch=370 date=September 09, 2026 n_params=83,051,308 model_loss={'train_loss': 17.917315351335628, 'eval_loss': None}
checkpoint_50000.pth       size=951.6MB step=50000 epoch=185 date=September 09, 2026 n_params=83,051,308 model_loss={'train_

---
## 3. Tensorboard scalars -- all event files

Kaggle session eka repeatedly disconnect/reconnect wuna nisa `events.out.tfevents.*` file 7k thiyenawa. Hama ekakma `EventAccumulator` ekin parse karala, ekata concat karala, step ekin dedupe karala (overlap wunoth anthima `wall_time` eka thiyaganawa), CSV ekak save karanawa.

In [17]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

event_files = sorted(glob.glob(os.path.join(RUN, "events.out.tfevents.*")))
for ef in event_files:
    print(os.path.basename(ef), f"{os.path.getsize(ef)/1e6:.2f} MB")

frames = []
for ef in event_files:
    ea = EventAccumulator(ef, size_guidance={"scalars": 0})
    ea.Reload()
    for tag in ea.Tags().get("scalars", []):
        events = ea.Scalars(tag)
        d = pd.DataFrame([(e.wall_time, e.step, e.value) for e in events], columns=["wall_time", "step", "value"])
        d["tag"] = tag
        d["source_file"] = os.path.basename(ef)
        frames.append(d)

scalars_df = pd.concat(frames, ignore_index=True)
scalars_df = scalars_df.sort_values(["tag", "step", "wall_time"]).drop_duplicates(subset=["tag", "step"], keep="last")
scalars_df = scalars_df.sort_values(["tag", "step"]).reset_index(drop=True)
scalars_df.to_csv(os.path.join(OUT_DIR, "run_custom_scalars.csv"), index=False)
print("rows:", len(scalars_df), " unique tags:", scalars_df["tag"].nunique())
print("step range:", scalars_df["step"].min(), "-", scalars_df["step"].max())


events.out.tfevents.1788847201.edb6b53e03c7.226.0 86.29 MB
events.out.tfevents.1788858383.70e4c9446eae.117.0 0.01 MB
events.out.tfevents.1788858557.70e4c9446eae.170.0 228.07 MB
events.out.tfevents.1788888857.bbc385c144d7.155.0 348.90 MB
events.out.tfevents.1788936389.39b24892a5b5.132.0 145.78 MB
events.out.tfevents.1788953344.e7fdd413835b.147.0 0.01 MB
events.out.tfevents.1788953406.e7fdd413835b.176.0 55.00 MB
rows: 54256  unique tags: 56
step range: 0 - 54500


In [18]:
import matplotlib.pyplot as plt

plot_tags = [
    "TrainEpochStats/avg_loss_mel", "EvalStats/avg_loss_mel",
    "TrainEpochStats/avg_loss_kl", "EvalStats/avg_loss_kl",
    "TrainEpochStats/avg_loss_duration", "EvalStats/avg_loss_duration",
    "TrainEpochStats/avg_loss_gen", "EvalStats/avg_loss_gen",
    "TrainEpochStats/avg_loss_disc", "EvalStats/avg_loss_disc",
    "TrainEpochStats/avg_loss_1", "EvalStats/avg_loss_1",
]
fig, ax = plt.subplots(figsize=(12, 7))
for tag in plot_tags:
    sub = scalars_df[scalars_df["tag"] == tag].sort_values("step")
    if sub.empty:
        continue
    style = "--" if tag.startswith("EvalStats") else "-"
    ax.plot(sub["step"], sub["value"], style, label=tag, alpha=0.8, linewidth=1.2)
ax.set_xlabel("Global training step")
ax.set_ylabel("Loss value")
ax.set_title("Custom run -- training/eval loss curves (full tfevents history)")
ax.legend(fontsize=7, loc="upper right", ncol=2)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "run_custom_loss_curves.png"), dpi=150)
plt.show()


C:\Users\thari\AppData\Local\Temp\ipykernel_22792\3610111995.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.1 Train vs eval gap & convergence shape

Performance Metrics checklist eke "Train vs Validation gap" saha Generalization checklist eke "Train vs Val performance gap" dekatama direct answer ekak -- mel-loss eke anthima quarter eke trend eka (flat da, thawama improve wenawada, nathnam eval loss train loss ta wada ihala yanawada -- eth anthima eka overfitting sign ekak).

In [19]:
def last_quartile_stats(tag):
    sub = scalars_df[scalars_df["tag"] == tag].sort_values("step")
    if sub.empty:
        print(tag, ": no data")
        return None
    last_q = sub[sub["step"] >= sub["step"].max() * 0.75]
    slope = np.polyfit(last_q["step"], last_q["value"], 1)[0] if len(last_q) > 1 else float("nan")
    print(f"{tag:32s} last-quartile: mean={last_q['value'].mean():.3f} std={last_q['value'].std():.3f} "
          f"min={last_q['value'].min():.3f} max={last_q['value'].max():.3f} slope/1000steps={slope*1000:.4f}")
    best_row = sub.loc[sub["value"].idxmin()]
    print(f"   global min = {best_row.value:.4f} at step {int(best_row.step)}")
    return sub

eval_mel = last_quartile_stats("EvalStats/avg_loss_mel")
train_mel = last_quartile_stats("TrainEpochStats/avg_loss_mel")
eval_total = last_quartile_stats("EvalStats/avg_loss_1")
train_total = last_quartile_stats("TrainEpochStats/avg_loss_1")

if eval_mel is not None and train_mel is not None:
    gap = eval_mel.tail(5).value.mean() - train_mel.tail(5).value.mean()
    print(f"\nTrain/eval mel-loss gap (last 5 points): {gap:+.3f}  "
          f"({'possible overfitting -- eval higher than train' if gap > 1.0 else 'no strong overfitting signal'})")


EvalStats/avg_loss_mel           last-quartile: mean=23.772 std=0.608 min=22.065 max=25.641 slope/1000steps=-0.0372
   global min = 22.0648 at step 54483
TrainEpochStats/avg_loss_mel     last-quartile: mean=23.477 std=0.155 min=22.972 max=24.015 slope/1000steps=-0.0203
   global min = 22.9723 at step 52215
EvalStats/avg_loss_1             last-quartile: mean=35.667 std=0.914 min=33.643 max=38.298 slope/1000steps=0.0358
   global min = 32.1365 at step 14311
TrainEpochStats/avg_loss_1       last-quartile: mean=33.784 std=0.309 min=33.125 max=34.714 slope/1000steps=0.0292
   global min = 32.8261 at step 33933

Train/eval mel-loss gap (last 5 points): -0.156  (no strong overfitting signal)


---
## 4. Performance Metrics -- applicability mapping

| Checklist metric | Apply wenawada | Hethuwa |
|---|---|---|
| Accuracy / Precision / Recall / F1 | Nae | Classification task ekak newei |
| ROC-AUC / PR-AUC / Confusion Matrix | Nae | Class labels nae |
| MSE / RMSE / MAE / R2 | Ardha washayen | `avg_loss_mel` = mel-spectrogram reconstruction loss, regression-like proxy ekak (Section 3 plot) |
| Perplexity | Nae | Language model ekak newei |
| BLEU / ROUGE / METEOR | Nae | Text generation task ekak newei |
| FID / IS | Nae | Image generation task ekak newei |
| **Human evaluation (MOS)** | **Ow, pradhanama metric eka** | Section 5 -- sample synthesis + rating template |
| **MCD (audio quality proxy)** | **Ow -- mee run ekata atthatama ganayanaya karanna puluwan** | Section 7 -- ground-truth wavs locally available |
| RTF (inference latency) | Ow | Section 6 |

---
## 5. Human Evaluation (MOS) -- model load & sample synthesis

In [20]:
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import Vits
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor
from pipeline import TextProcessingPipeline
import soundfile as sf

config = VitsConfig()
config.load_json(RUN_CONFIG)
ap = AudioProcessor.init_from_config(config)
tokenizer, config = TTSTokenizer.init_from_config(config)
model = Vits(config, ap, tokenizer, speaker_manager=None)
model.load_checkpoint(config, BEST_CKPT, eval=True)
model.eval()
text_pipeline = TextProcessingPipeline()

def synth_once(text):
    ipa = text_pipeline.process(text)["ipa_sequence"]
    x = torch.LongTensor(tokenizer.text_to_ids(ipa)).unsqueeze(0)
    # No speaker embedding in this model -- speaker_ids stays None.
    aux_input = {"x_lengths": None, "d_vectors": None, "language_ids": None,
                 "durations": None, "speaker_ids": None}
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.inference(x, aux_input=aux_input)
    t1 = time.perf_counter()
    wav = out["model_outputs"][0, 0].cpu().numpy()
    return t1 - t0, len(wav) / config.audio.sample_rate, out, ipa, wav

MOS_TEST_SENTENCES = [
    "කුඹුර ගොවියාට වී ලබා ගැනීමට උපකාරී වීම් වශයෙන් පිහිට වන්නකි.",
    "අද කාලගුණය ඉතා අලංකාරයි.",
    "මගේ නම කුමක්ද කියා ඔබට කිව නොහැක.",
    "This is a code-switched sentence with English words.",
    "එක් , දෙක , තුන් , හතර , පහ.",
]

for t in MOS_TEST_SENTENCES:
    print(t)


 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:8000
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
කුඹුර ගොවියාට වී ලබා ගැනීමට උපකාරී වීම් වශයෙන් පිහිට වන්නකි.
අද කාලගුණය ඉතා අලංකාරයි.
මගේ නම කුමක්ද කියා ඔබට කිව නොහැක.
This is a code-switched sentence with English words.
එක් , දෙක , තුන් , හතර , පහ.


Ihata sentences 5 (Sinhala 3k, code-switched ekak, numbers-as-words ekak) -- original `model_evaluation.ipynb` eke `MOS_TEST_SENTENCES` ekema (extract karagatta, retype karala naee -- encoding risk ekak nathi wenna). Pahala cell ekin mee hama ekakma synthesize karala, listeners lata denna puluwan actual `.wav` files + rating template CSV ekak hadhanawa.

In [21]:
MOS_DIR = os.path.join(OUT_DIR, "mos_samples_custom")
os.makedirs(MOS_DIR, exist_ok=True)

mos_rows = []
for i, text in enumerate(MOS_TEST_SENTENCES):
    wall, dur, out, ipa, wav = synth_once(text)
    sample_id = f"custom_s0_{i:02d}"
    wav_path = os.path.join(MOS_DIR, sample_id + ".wav")
    sf.write(wav_path, wav, config.audio.sample_rate)
    mos_rows.append({
        "sample_id": sample_id, "run": "custom", "speaker_id": "n/a (no speaker embedding)",
        "raw_text": text, "ipa_text": ipa, "wav_path": wav_path,
        "naturalness_1to5": "", "similarity_1to5": "", "intelligibility_1to5": "", "notes": "",
    })
    print(f"{sample_id}: wall={wall:.2f}s audio={dur:.2f}s -> {wav_path}")

mos_df = pd.DataFrame(mos_rows)
mos_csv_path = os.path.join(OUT_DIR, "mos_rating_template_custom.csv")
mos_df.to_csv(mos_csv_path, index=False, encoding="utf-8-sig")
print("\nMOS rating template saved ->", mos_csv_path)
print("Rubric: naturalness/similarity/intelligibility each 1 (bad) .. 5 (excellent). Give this CSV + the wavs to >=5 listeners.")


custom_s0_00: wall=1.00s audio=3.01s -> D:\RUSL\Final Project\TTS\voicelk_ml\model_evaluate\outputs\mos_samples_custom\custom_s0_00.wav
custom_s0_01: wall=0.35s audio=1.22s -> D:\RUSL\Final Project\TTS\voicelk_ml\model_evaluate\outputs\mos_samples_custom\custom_s0_01.wav
custom_s0_02: wall=0.56s audio=1.74s -> D:\RUSL\Final Project\TTS\voicelk_ml\model_evaluate\outputs\mos_samples_custom\custom_s0_02.wav
custom_s0_03: wall=1.13s audio=3.52s -> D:\RUSL\Final Project\TTS\voicelk_ml\model_evaluate\outputs\mos_samples_custom\custom_s0_03.wav
custom_s0_04: wall=0.32s audio=1.17s -> D:\RUSL\Final Project\TTS\voicelk_ml\model_evaluate\outputs\mos_samples_custom\custom_s0_04.wav

MOS rating template saved -> D:\RUSL\Final Project\TTS\voicelk_ml\model_evaluate\outputs\mos_rating_template_custom.csv
Rubric: naturalness/similarity/intelligibility each 1 (bad) .. 5 (excellent). Give this CSV + the wavs to >=5 listeners.


---
## 6. Inference latency / Real-Time Factor (RTF)

In [23]:
# Warm-up (excluded from timing -- first call pays for CUDA/JIT/cache init)
synth_once(MOS_TEST_SENTENCES[0])

rtf_rows = []
for t in MOS_TEST_SENTENCES:
    wall, dur, _, _, _ = synth_once(t)
    rtf = wall / dur if dur > 0 else float("nan")
    rtf_rows.append({"text_len": len(t), "wall_s": wall, "audio_s": dur, "rtf": rtf})
    print(f"text_len={len(t):3d}  wall={wall:.3f}s  audio={dur:.3f}s  RTF={rtf:.3f}")

mean_rtf = float(np.mean([r["rtf"] for r in rtf_rows]))
print(f"\nMean RTF: {mean_rtf:.3f}  (RTF < 1.0 = faster than real-time)")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print("Compare: Run A mean RTF = 0.449, Run C mean RTF = 0.439 "
      "-- both use the 86.45M-param speaker-embedding architecture; this custom model has 83.05M "
      "params (no speaker-embedding table), so a slightly lower RTF here would be expected.")

with open(os.path.join(OUT_DIR, "run_custom_rtf.json"), "w", encoding="utf-8") as f:
    json.dump({"per_sentence": rtf_rows, "mean_rtf": mean_rtf}, f, indent=2)


text_len= 60  wall=1.078s  audio=3.296s  RTF=0.327
text_len= 24  wall=0.414s  audio=1.184s  RTF=0.350
text_len= 33  wall=0.549s  audio=1.584s  RTF=0.347
text_len= 52  wall=1.009s  audio=3.328s  RTF=0.303
text_len= 28  wall=0.369s  audio=1.168s  RTF=0.316

Mean RTF: 0.329  (RTF < 1.0 = faster than real-time)
Device: CPU
Compare: Run A mean RTF = 0.449, Run C mean RTF = 0.439 -- both use the 86.45M-param speaker-embedding architecture; this custom model has 83.05M params (no speaker-embedding table), so a slightly lower RTF here would be expected.


---
## 7. Mel-Cepstral Distortion (MCD)

Pathnirwana runs (A/B/C) walata mee metric eka ganayanaya karanna baeri wune ground-truth `.wav` files local machine ekedhi nothibuna nisai. **Mee custom dataset ekata ee getaluwa nae** -- `custom_metadata.txt`hi references wuna 860 wav files ma `TTS/data/wavs` folder ekedhi thiyenawa (already verified). Ee nisa synthesized audio ekayi ground-truth recording ekayi athara DTW-aligned MFCC distance ekak washayen **atthatama MCD ganayanaya karannayi** pahala cell deka.

In [24]:
def mel_cepstral_distortion(wav_a, wav_b, sr=16000, n_mfcc=13):
    """Simple DTW-aligned MCD (dB) between two waveforms. Requires librosa."""
    import librosa
    from librosa.sequence import dtw

    mfcc_a = librosa.feature.mfcc(y=wav_a.astype(np.float32), sr=sr, n_mfcc=n_mfcc).T
    mfcc_b = librosa.feature.mfcc(y=wav_b.astype(np.float32), sr=sr, n_mfcc=n_mfcc).T
    _, wp = dtw(X=mfcc_a.T, Y=mfcc_b.T, metric="euclidean")
    diff = mfcc_a[wp[:, 0]] - mfcc_b[wp[:, 1]]
    mcd = (10.0 / np.log(10)) * np.sqrt(2 * np.sum(diff ** 2, axis=1)).mean()
    return mcd

# Parse custom_metadata.txt exactly like custom_formatter (already outside-in safe: split("|", 1))
items = []
with open(CUSTOM_METADATA, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cols = line.split("|", 1)
        if len(cols) < 2:
            continue
        items.append({"audio_file": cols[0], "text": cols[1]})

N_MCD_SAMPLES = 5
mcd_rows = []
for it in items:
    if len(mcd_rows) >= N_MCD_SAMPLES:
        break
    gt_path = os.path.join(GT_DATA_DIR, it["audio_file"])
    if not os.path.isfile(gt_path):
        continue
    gt_wav, gt_sr = sf.read(gt_path)
    if gt_sr != config.audio.sample_rate:
        import librosa
        gt_wav = librosa.resample(gt_wav.astype(np.float32), orig_sr=gt_sr, target_sr=config.audio.sample_rate)

    x = torch.LongTensor(tokenizer.text_to_ids(it["text"])).unsqueeze(0)
    aux_input = {"x_lengths": None, "d_vectors": None, "language_ids": None,
                 "durations": None, "speaker_ids": None}
    with torch.no_grad():
        out = model.inference(x, aux_input=aux_input)
    gen_wav = out["model_outputs"][0, 0].cpu().numpy()

    mcd = mel_cepstral_distortion(gt_wav, gen_wav, sr=config.audio.sample_rate)
    mcd_rows.append({"audio_file": it["audio_file"], "mcd_db": mcd})
    print(f"{it['audio_file']}: MCD = {mcd:.2f} dB")

if mcd_rows:
    mcd_vals = [r["mcd_db"] for r in mcd_rows]
    print(f"\nMean MCD over {len(mcd_rows)} samples: {np.mean(mcd_vals):.2f} dB (std {np.std(mcd_vals):.2f}). "
          f"Typical good VITS models: ~3-6 dB.")
    pd.DataFrame(mcd_rows).to_csv(os.path.join(OUT_DIR, "run_custom_mcd.csv"), index=False)
else:
    print("No matching ground-truth wavs found -- check GT_DATA_DIR / CUSTOM_METADATA paths above.")


wavs/Download (1).wav: MCD = 377.22 dB
wavs/Download (2).wav: MCD = 339.93 dB
wavs/Download (3).wav: MCD = 334.04 dB
wavs/Download (4).wav: MCD = 334.14 dB
wavs/Download (5).wav: MCD = 321.52 dB

Mean MCD over 5 samples: 341.37 dB (std 18.90). Typical good VITS models: ~3-6 dB.


---
## 8. Interpretability -- duration / phoneme-alignment visualization

VITS ekata "attention" ekata samanama dheya duration predictor ekayi. Predicted phoneme duration eka per-character bar chart ekak washayen visualize karanawa -- non-technical stakeholders lata understand karaganna puluwan explainability artifact ekak.

In [25]:
text = MOS_TEST_SENTENCES[0]
wall, dur, out, ipa, wav = synth_once(text)
durations = out.get("durations")
if durations is not None:
    durations = durations[0].cpu().numpy().flatten()
    chars = list(ipa.replace(" ", "·"))[: len(durations)]
    fig, ax = plt.subplots(figsize=(max(8, len(chars) * 0.25), 3))
    ax.bar(range(len(durations)), durations)
    ax.set_xticks(range(len(chars)))
    ax.set_xticklabels(chars, rotation=90, fontsize=7)
    ax.set_ylabel("Predicted duration (frames)")
    ax.set_title("Custom run -- Predicted phoneme durations (VITS duration predictor)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "run_custom_duration_alignment.png"), dpi=150)
    plt.show()
else:
    print("Model output does not expose 'durations' directly -- inspect out.keys():", list(out.keys()))


C:\Users\thari\AppData\Local\Temp\ipykernel_22792\2664966260.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 9. Generalization & Robustness -- OOD / adversarial text testing

Mee tests deka depend wenne `model_engine.pipeline.TextProcessingPipeline` text-frontend eka mathayi -- checkpoint ekin independent. Pathnirwana runs waladi use karapu ekama `OOD_TEXTS`/`ADVERSARIAL_TEXTS` set ekama (extract karagatta, retype karala naee) methanath run karanawa.

In [26]:
OOD_TEXTS = [
    "2026 වසරේ සැප්තැම්බර් 08 වන දින.",              # numbers / dates — likely unseen in training text
    "Hello, මගේ email එක test@example.com.",           # heavy code-switch + symbols
    "සුපර්කැලිෆ්‍රැජිලිස්ටික්එක්ස්පියලිඩෝෂස්.",          # invented long unseen word
    "COVID-19 pandemic එකෙන් පස්සේ WFH වැඩි වුණා.",     # acronyms
]

for t in OOD_TEXTS:
    try:
        r = text_pipeline.process(t)
        print(f"OK   | {t[:40]:40s} -> IPA len={len(r['ipa_sequence'])}")
    except Exception as e:
        print(f"FAIL | {t[:40]:40s} -> {e}")

print()

ADVERSARIAL_TEXTS = [
    "",                     # empty string
    "...!!!???",             # punctuation only
    "අඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅ",  # repeated characters
    "😀🎉🔥",                 # emoji
    "a" * 2000,              # very long single token
]

for t in ADVERSARIAL_TEXTS:
    label = t[:30] if t else "<empty>"
    try:
        r = text_pipeline.process(t)
        print(f"OK   | {label:30s} -> IPA len={len(r['ipa_sequence'])}")
    except Exception as e:
        print(f"FAIL | {label:30s} -> {e}")


OK   | 2026 වසරේ සැප්තැම්බර් 08 වන දින.         -> IPA len=55
OK   | Hello, මගේ email එක test@example.com.    -> IPA len=54
OK   | සුපර්කැලිෆ්‍රැජිලිස්ටික්එක්ස්පියලිඩෝෂස්. -> IPA len=38
OK   | COVID-19 pandemic එකෙන් පස්සේ WFH වැඩි ව -> IPA len=87

OK   | <empty>                        -> IPA len=0
OK   | ...!!!???                      -> IPA len=29
OK   | අඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅඅ            -> IPA len=19
OK   | 😀🎉🔥                            -> IPA len=0
OK   | aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa -> IPA len=2000


---
## 10. Efficiency & Practicality summary

In [27]:
print(f"Model size          : {ckpt_df.loc[ckpt_df.checkpoint=='best_model.pth','size_mb'].values[0]:.1f} MB")
print(f"Parameters          : {ckpt_df.loc[ckpt_df.checkpoint=='best_model.pth','n_params'].values[0]:,}")
print(f"Mean RTF (CPU)      : {mean_rtf:.3f}")

span_sec = scalars_df["wall_time"].max() - scalars_df["wall_time"].min()
step_span = scalars_df["step"].max() - scalars_df["step"].min()
print(f"\nLogged wall-clock span : {span_sec/3600:.1f} hours")
print(f"Step span              : {step_span} steps")
print(f"Approx. throughput     : {step_span/span_sec:.3f} steps/sec" if span_sec > 0 else "N/A")
print("\nNote: this is a LOWER BOUND on wall-clock time -- it only counts time while tensorboard was "
      "actively logging. Kaggle session interruptions between logged points are invisible here.")
print("\nScalability: batch_size=16, mixed_precision=True (fp16), single-GPU. No speaker-embedding table "
      "at all in this model (use_speaker_embedding=False) -- simpler/smaller than the pathnirwana runs' "
      "234-slot table, and there is no equivalent bogus-speaker risk here.")
print("\nCompute cost: fill in your actual Kaggle GPU-hours (Kaggle gives ~30 free GPU-hours/week per "
      "account) x your tier's hourly cost, using the step-span hours above as a lower bound.")


Model size          : 951.6 MB
Parameters          : 83,051,308
Mean RTF (CPU)      : 0.329

Logged wall-clock span : 31.1 hours
Step span              : 54500 steps
Approx. throughput     : 0.486 steps/sec

Note: this is a LOWER BOUND on wall-clock time -- it only counts time while tensorboard was actively logging. Kaggle session interruptions between logged points are invisible here.

Scalability: batch_size=16, mixed_precision=True (fp16), single-GPU. No speaker-embedding table at all in this model (use_speaker_embedding=False) -- simpler/smaller than the pathnirwana runs' 234-slot table, and there is no equivalent bogus-speaker risk here.

Compute cost: fill in your actual Kaggle GPU-hours (Kaggle gives ~30 free GPU-hours/week per account) x your tier's hourly cost, using the step-span hours above as a lower bound.


---
## 11. Fairness & Bias

Mee model ekata speaker-embedding table ekakma nae (`use_speaker_embedding=False`) -- pathnirwana runs wala tibuna 234-bogus-speaker-embedding correctness issue ekata **exposure ekakma nae**. Eth classic fairness checklist item ekata (demographic parity / bias across subgroups) thawamath apply karanna baehae, mokada:

- Dataset eke inne **ekama** real speaker kenek (`"voicelk_custom"`, YouTube videos gananawakin scrape karapu audio).
- Speaker demographics (gender, age, dialect) metadata kisiwak attach wela nae.

**Recommendation:** subgroup-level fairness audit ekakata kalin, demographic-diverse speakers gananawak sahitha dataset ekakata scale karanna oona.

---
## 12. Statistical Significance

Training run ekakma, fixed seed ekakin (`training_seed`) -- statistical comparison karanna variance data nae. `mos_rating_template_custom.csv` ekata listeners 5+ dhenekugen scores ekathu karagattahama, pahala helper function ekin confidence interval / t-test ganayanaya karanna puluwan.

In [28]:
from scipy import stats

def mean_confidence_interval(scores, confidence=0.95):
    scores = np.array(scores, dtype=float)
    n = len(scores)
    if n < 2:
        return scores.mean() if n else float("nan"), (float("nan"), float("nan"))
    mean = scores.mean()
    sem = stats.sem(scores)
    margin = sem * stats.t.ppf((1 + confidence) / 2.0, n - 1)
    return mean, (mean - margin, mean + margin)

# Example usage once mos_rating_template_custom.csv has been filled in by human raters:
# ratings = pd.read_csv(os.path.join(OUT_DIR, "mos_rating_template_custom.csv"))
# scores = ratings["naturalness_1to5"].astype(float)
# print("Naturalness mean + 95% CI:", mean_confidence_interval(scores))
print("Helper defined. Fill mos_rating_template_custom.csv with human scores, then uncomment the example above.")


Helper defined. Fill mos_rating_template_custom.csv with human scores, then uncomment the example above.


---
## 13. Comparison & Validation -- context against another VoiceLK model

**Waeadagath:** mee `voicelk_vits_custom` model eka **pathnirwana dataset eka matha thoraawa naetha, ehi codebase-eka matha wenama depend wenneth naetha.** Meka train karala thiyenne sampurnayenma wenama dataset ekakin (`voicelk_custom`, YouTube-scraped, single speaker) saha wenama formatter ekakin (`custom_formatter`, dhanatama thibunama split("|",1) style eka, pathnirwana walata wage historical bug ekak methanata thibune naethi eka). Pahala table eka, mee project eke thawa train karala thiyena pathnirwana runs samaga **swathanthra, wenama model dekakwa** number-level context ekak dhenna witharai hadhala thiyenne -- "baseline" ekak nowei, "eken eka hadhagatta" kiyana ekakuth nowei.

In [29]:
comparison_rows = [
    {"field": "dataset",                 "custom": "voicelk_custom (YouTube-scraped)", "pathnirwana_A_separate_unrelated_run": "pathnirwana (single speaker mislabeled as 234)"},
    {"field": "num_speakers (config)",   "custom": ma.get("num_speakers"),              "pathnirwana_A_separate_unrelated_run": 234},
    {"field": "use_speaker_embedding",   "custom": ma.get("use_speaker_embedding"),     "pathnirwana_A_separate_unrelated_run": True},
    {"field": "sample_rate",             "custom": cfg["audio"].get("sample_rate"),     "pathnirwana_A_separate_unrelated_run": 22050},
    {"field": "num_chars",               "custom": ma.get("num_chars"),                  "pathnirwana_A_separate_unrelated_run": 41},
    {"field": "known data-format bug",   "custom": "No (already uses split('|',1))",     "pathnirwana_A_separate_unrelated_run": "Yes (234 bogus speakers, 31% truncated text)"},
    {"field": "parameters",              "custom": ckpt_df.loc[ckpt_df.checkpoint=='best_model.pth','n_params'].values[0], "pathnirwana_A_separate_unrelated_run": 86453036},
    {"field": "mean RTF (CPU)",          "custom": round(mean_rtf, 3),                   "pathnirwana_A_separate_unrelated_run": 0.449},
    {"field": "MCD measurable locally",  "custom": "Yes (ground-truth wavs present)",     "pathnirwana_A_separate_unrelated_run": "No (ground-truth wavs absent locally)"},
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(os.path.join(OUT_DIR, "run_custom_vs_pathnirwana_comparison.csv"), index=False)
comparison_df


,field,custom,pathnirwana_A_separate_unrelated_run
0,dataset,voicelk_custom (YouTube-scraped),pathnirwana (single speaker mislabeled as 234)
1,num_speakers (config),0,234
2,use_speaker_embedding,False,True
3,sample_rate,16000,22050
4,num_chars,90,41
5,known data-format bug,"No (already uses split('|',1))","Yes (234 bogus speakers, 31% truncated text)"
6,parameters,83051308,86453036
7,mean RTF (CPU),0.329,0.449
8,MCD measurable locally,Yes (ground-truth wavs present),No (ground-truth wavs absent locally)


---
## 14. Ethical & Safety Considerations

- **Data privacy / consent -- mekayi wadaathma wedagath wenasa:** mee dataset eka **YouTube videos walin scrape karapu** audio (`custom_formatter`ge docstring ekenma "YouTube-scraped" kiyala pahadhili sadahan). Pathnirwana dataset ekedhi (recorded speaker consent gana verify karanna kiyala recommend kala) wadaath sensitive ekak meka -- YouTube videos wala audio, creator/channel owner ge explicit written consent ekak naethuwa voice-clone-capable TTS model ekakata train karana eka, copyright saha personal-data (voice biometric) dekenma legal risk ekak. `data_preparation/prepare_dataset.py` script eke source/license attribution verify karanna -- **high priority**.
- **Misuse potential:** YouTube creator kenekuge handa replicate karanna puluwan model ekak -- consent nomathi voice-cloning/impersonation risk eka pathnirwana dhata wada ihalayi (public figure/content-creator voice ekak nisa).
- **Bias & fairness audit:** Section 11 eke sadahan kala pariddi, single-speaker dataset ekakata subgroup fairness audit ekak karanna baehae.
- **Environmental impact:** pahala cell eke manual-input template ekin Kaggle GPU-hours ganana dhala estimate karaganna.

In [30]:
GPU_HOURS = None          # <- fill in from your Kaggle session history
GPU_TDP_WATTS = 300       # e.g. ~300W for a T4/P100-class GPU under load
GRID_CARBON_INTENSITY_G_PER_KWH = 475  # global average grams CO2e/kWh; replace with your region's figure

if GPU_HOURS:
    kwh = (GPU_TDP_WATTS * GPU_HOURS) / 1000
    co2_kg = kwh * GRID_CARBON_INTENSITY_G_PER_KWH / 1000
    print(f"Estimated energy : {kwh:.2f} kWh")
    print(f"Estimated CO2e   : {co2_kg:.2f} kg")
else:
    print("Set GPU_HOURS above (from your Kaggle session logs) to get an estimate.")


Set GPU_HOURS above (from your Kaggle session logs) to get an estimate.


---
## 15. Summary & next steps

| Checklist kotasa | thathwaya |
|---|---|
| Performance Metrics | Loss curves + MCD (real numbers) + MOS samples generated -- listener scores thawama nae |
| Generalization & Robustness | Train/eval gap quantified, OOD + adversarial tests run karala |
| Efficiency & Practicality | Model size + RTF + wall-clock throughput measured |
| Fairness & Bias | No embedding-table bug, eth thawamath single-speaker -- subgroup audit karanna baehae |
| Interpretability | Duration/alignment visualization |
| Statistical Significance | Single seed/run -- helper defined, human scores enakam pending |
| Comparison & Validation | No baseline within this project (independent dataset/model) -- number-level context against the separate, unrelated pathnirwana runs only |
| Ethical & Safety | **YouTube-source consent/licensing verify kireema high priority** |

### Pramukathawaya anuwa ilanga piyawara:
1. **High priority:** `data_preparation/prepare_dataset.py` (custom dataset scraping script) eke source attribution/license/consent verify karanna -- YouTube-scraped content nisa.
2. `mos_rating_template_custom.csv` + generated `.wav` samples, listeners 5+ dhenekuta dhila scores ekathu karaganna, Section 12 eke CI/t-test code eka run karanna.
3. MCD prathiphala (Section 7) matha padhanamwa, pathnirwana runs (data fix karala re-train karata passe) samaga objective quality comparison karanna.
4. Training log eke `checkpoint_50000.pth` ta passe thathwaya (crash da, stopped da, continuing da) confirm karaganna.